# Frameworks 02 - Strands (CLI)

Notebook CLI paralelo a `tutorials/frameworks/02_aws_strands.ipynb`.

**Objetivo:** Inspeccionar Strands sobre los cinco Providers.

Este notebook no llama factories de Agentic Systems directamente: ejecuta el
entrypoint CLI real, conserva la salida Rich y valida después el JSON del mismo
contrato.


## Cómo se ejecuta

La forma portable es `python -m agentic_systems.cli ...`. Después de instalar
el wheel, el entrypoint equivalente es `agentic-systems ...`.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def _repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio.")


ROOT = _repo_root()
CLI = [sys.executable, "-m", "agentic_systems.cli"]


def run_cli(*args: str, expected: int = 0) -> str:
    env = os.environ.copy()
    source_path = str(ROOT / "src")
    env["PYTHONPATH"] = (
        source_path
        if not env.get("PYTHONPATH")
        else source_path + os.pathsep + env["PYTHONPATH"]
    )
    command = [*CLI, *args]
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        check=False,
    )
    print("$ " + " ".join(command))
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    assert completed.returncode == expected, completed.stderr
    return completed.stdout


def run_cli_json(*args: str) -> dict:
    return json.loads(run_cli(*args, "--json"))


def assert_rich(output: str, title: str) -> None:
    assert title in output
    ascii_box = "+" in output and "|" in output
    unicode_box = "─" in output and "│" in output
    assert ascii_box or unicode_box


## 1) Salida humana Rich

La celda conserva stdout y comprueba título y bordes. Esto detecta tablas o
paneles truncados, además del exit code.


In [2]:
rich_output = run_cli(*['matrix', 'check', '--framework', 'strands'])
assert_rich(rich_output, 'Matrix Workflow')


$ C:\Python314\python.exe -m agentic_systems.cli matrix check --framework strands
+------------------------------ Matrix Workflow ------------------------------+
| {                                                                           |
|   "combination_count": 5,                                                   |
|   "failed": 0,                                                              |
|   "framework_filter": "strands",                                            |
|   "live": false,                                                            |
|   "not_run": 4,                                                             |
|   "passed": 1,                                                              |
|   "provider_filter": null,                                                  |
|   "results": [                                                              |
|     {                                                                       |
|       "execution": "passed",        

## 2) Contrato de máquina

La misma ruta se ejecuta con `--json` para afirmar campos y cardinalidad sin
parsear la presentación Rich.


In [3]:
args = ['matrix', 'check', '--framework', 'strands']
live_flag = os.getenv("RUN_CLI_LIVE", "0").strip().lower() in {"1", "true", "yes"}
if live_flag:
    args.append("--live")
    args.append("--require-pass")
payload = run_cli_json(*args)

assert payload["combination_count"] == 5
assert payload["failed"] == 0
assert len(payload["results"]) == 5
if live_flag:
    assert payload["passed"] == 5
payload


$ C:\Python314\python.exe -m agentic_systems.cli matrix check --framework strands --json
{
  "combination_count": 5,
  "failed": 0,
  "framework_filter": "strands",
  "live": false,
  "not_run": 4,
  "passed": 1,
  "provider_filter": null,
  "results": [
    {
      "execution": "passed",
      "framework": "strands",
      "offline_certified": true,
      "provider": "python-runtime",
      "ready": true,
      "reason": "Offline contract certified and runtime signals available.",
      "result": {
        "data": {
          "value": "ok"
        },
        "engine": "python-runtime",
        "final": {
          "value": "ok"
        },
        "mode": "eval",
        "ok": true,
        "text": "{\"value\": \"ok\"}\n",
        "tool_outputs": [
          {
            "input": {
              "value": "ok"
            },
            "ok": true,
            "output": [
              {
                "text": "{\"value\": \"ok\"}"
              }
            ],
            "tool": "c

{'combination_count': 5,
 'failed': 0,
 'framework_filter': 'strands',
 'live': False,
 'not_run': 4,
 'passed': 1,
 'provider_filter': None,
 'results': [{'execution': 'passed',
   'framework': 'strands',
   'offline_certified': True,
   'provider': 'python-runtime',
   'ready': True,
   'reason': 'Offline contract certified and runtime signals available.',
   'result': {'data': {'value': 'ok'},
    'engine': 'python-runtime',
    'final': {'value': 'ok'},
    'mode': 'eval',
    'ok': True,
    'text': '{"value": "ok"}\n',
    'tool_outputs': [{'input': {'value': 'ok'},
      'ok': True,
      'output': [{'text': '{"value": "ok"}'}],
      'tool': 'cli_echo'}],
    'tools_called': ['cli_echo'],
    'usage': {'scheduler': {'attempts': 1,
      'latency_ms': 1839.387,
      'retries': 0,
      'timed_out': False}},
    'validation_ok': True},
   'status': 'ready'},
  {'execution': 'not-run',
   'execution_reason': 'Pass --live to cross an external provider boundary.',
   'framework': '

## Resultado e interpretación

Rich responde a lectura humana; JSON responde a automatización. Ambos nacen del
mismo comando y del mismo escenario público. Un estado `not-run` conserva el
motivo, pero no cuenta como evidencia live.
